<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/03_check_prompt_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inspect a case file

This notebook explores one of the case files written by `02_make_prompt_files.ipynb` into `model_inputs/ri_prediction/`: the development set (20 RI + 20 no-RI storms, 2016) or the test set (50 RI + 50 no-RI storms, 2017–2025). Pick the file in the Load cell. Each line of the file is one observation ("case") with these fields:

| field | meaning |
|---|---|
| `case_id` | unique key: storm ID + 3-digit observation number |
| `split` | `dev` or `test` |
| `sid`, `name`, `season`, `basin` | the storm (bookkeeping only; never shown to the model) |
| `iso_time` | time of the current (last) observation in the prompt |
| `obs_index` | 0-based position of the current observation in its storm |
| `hours_since_start` | hours since the storm's first observation |
| `storm_group` | `ri` (storm has at least one RI event) or `no_ri` |
| `ri_tplus24` | true label: 1 if `usa_wind` rises by 30 kt or more over the next 24 hours |
| `image_t_core`, `image_t_mid`, `image_t_wide` | paths to the satellite image at time t in three crop sizes, relative to `model_inputs/` (always filled in; see `model_inputs/images/manifest.csv` for which images exist) |
| `prompt` | the full prompt sent to the LLM (no storm position, ID, name, season or date) |

## Setup

The next cell works out the repo folder and makes the code in `src/` importable, so the notebook runs the same way locally and in Colab. In Colab it first clones this repo and moves into it.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

## Load

In [ ]:
import json
import re
import pandas as pd

pd.set_option("display.max_colwidth", 80)

# Pick the file to inspect: the development set or the test set.
PATH = ROOT / "model_inputs" / "ri_prediction" / "test" / "test_2017-2025_50ri_50nori_text_only.jsonl.gz"
# PATH = ROOT / "model_inputs" / "ri_prediction" / "dev" / "dev_2016_20ri_20nori_text_only.jsonl.gz"
TARGET = "ri_tplus24"

assert PATH.exists(), f"{PATH.relative_to(ROOT)} not found. Run 02_make_prompt_files.ipynb, or update this clone with git pull."

# dtype=str keeps the basin code "NA" (North Atlantic) as text instead of a missing value;
# convert_dates=False keeps iso_time as text (pandas otherwise turns *_time columns into dates).
d = pd.read_json(PATH, lines=True, convert_dates=False, dtype={"sid": str, "iso_time": str, "basin": str, "case_id": str})
print(f"{PATH.name}: {PATH.stat().st_size / 1e6:.1f} MB compressed, {len(d)} cases, {d['sid'].nunique()} storms")
d.drop(columns="prompt").head()

## Derived columns

Some useful values are pulled out of each prompt:

- `prompt_chars`: prompt length in characters.
- `prompt_tokens_est`: a rough token estimate (characters / 3.5). Use your model's tokenizer for exact counts.
- `n_history_rows`: number of observation records inside `<storm_history>`.

In [ ]:
HISTORY_RE = re.compile(r"<storm_history>\n(.*?)\n</storm_history>", re.DOTALL)

def history_records(prompt):
    """The list of observation dicts inside the prompt's <storm_history> block."""
    return json.loads(HISTORY_RE.search(prompt).group(1))

d["prompt_chars"] = d["prompt"].str.len()
d["prompt_tokens_est"] = (d["prompt_chars"] / 3.5).round().astype(int)
d["n_history_rows"] = d["prompt"].map(lambda p: len(history_records(p)))

d.drop(columns="prompt").head()

## Sanity checks

These checks confirm the file is internally consistent:

1. Every placeholder in the template has been filled.
2. Each prompt's history has `obs_index + 1` rows.
3. Within a storm, `obs_index` runs 0, 1, 2, ... with no gaps.
4. Each prompt's history equals the previous prompt's history plus one new row (the history only grows by appending).
5. The text before `<storm_history>` is identical in every prompt.

In [ ]:
assert not d["prompt"].str.contains("{{", regex=False).any(), "unfilled placeholder"
assert (d["n_history_rows"] == d["obs_index"] + 1).all()
assert d.groupby("sid")["obs_index"].apply(lambda s: list(s) == list(range(len(s)))).all()

for sid, g in d.groupby("sid", sort=False):
    prev = None
    for p in g["prompt"]:
        hist = history_records(p)
        if prev is not None:
            assert hist[:-1] == prev, f"history of {sid} is not an append of the previous prompt"
        prev = hist

prefixes = d["prompt"].str.split("<storm_history>").str[0]
assert prefixes.nunique() == 1, "template text differs between prompts"

print("all checks passed | shared template prefix:", len(prefixes.iloc[0]), "chars")

### Nothing that identifies the storm

No prompt may contain the storm's position (`lat`, `lon`, `abs_lat`), in the history records or in the feature dictionary, nor its ID, name or date. Also checks that every case's image paths follow the naming rule, and counts how many of those images exist on this computer.

In [ ]:
LOCATION = {"lat", "lon", "abs_lat"}
FEATURE_DICT_RE = re.compile(r"<feature_dictionary>.*?(\{.*\})\s*</feature_dictionary>", re.DOTALL)

for _, case in d.iterrows():
    p = case["prompt"]
    assert not any(LOCATION & set(r) for r in history_records(p)), case["case_id"]
    assert not LOCATION & set(json.loads(FEATURE_DICT_RE.search(p).group(1))), case["case_id"]
    history_text = p.split("<storm_history>")[1]          # where the data enters the prompt
    identifiers = [case["sid"], case["iso_time"], case["iso_time"][:10]]
    if case["name"] not in ("UNNAMED", "NOT_NAMED"):
        identifiers.append(case["name"])
    assert not any(re.search(rf"\b{re.escape(x)}\b", history_text) for x in identifiers), case["case_id"]

assert d["case_id"].is_unique
print("no prompt contains the storm's position, ID, name or date")
for crop in ["core", "mid", "wide"]:
    assert (d[f"image_t_{crop}"] == "images/" + crop + "/" + d["case_id"] + "_" + crop + ".png").all(), crop
print("image files present on this computer:",
      {crop: int(d[f"image_t_{crop}"].map(lambda p: (ROOT / "model_inputs" / p).exists()).sum()) for crop in ["core", "mid", "wide"]},
      "of", len(d))

### Plot style

Shared settings for all plots: thin marks, light hairline grids, and neutral text. Two fixed colours are used throughout: blue for non-RI rows and orange for RI rows. The pair is colour-blind safe, and every two-series plot also has a legend.

In [ ]:
import matplotlib.pyplot as plt

C_NO_RI, C_RI = "#2a78d6", "#eb6834"                   # fixed colours: non-RI rows, RI rows
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 1, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "axes.labelcolor": INK_MUTED, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "legend.frameon": False, "figure.dpi": 110,
})

## Label balance

Each set is balanced by storm (equal numbers of RI and no-RI storms), but not by row. RI storms also have many non-RI hours, so most rows are labelled 0.

In [ ]:
print(d[TARGET].value_counts().rename("rows").to_frame().assign(share=lambda x: (x["rows"] / x["rows"].sum()).round(3)))
d.groupby("storm_group").agg(storms=("sid", "nunique"), rows=("sid", "size"), ri_rows=(TARGET, "sum"),
                             ri_share=(TARGET, "mean")).round(3)

## Prompt size

How long the prompts are overall, and how they grow as the history gets longer.

In [ ]:
d[["prompt_chars", "prompt_tokens_est", "n_history_rows"]].describe().round(0)

In [ ]:
# Prompt size by history length, in bins
bins = [0, 1, 5, 10, 20, 40, 80, 200]
d.groupby(pd.cut(d["n_history_rows"], bins), observed=True).agg(
    prompts=("sid", "size"), mean_tokens_est=("prompt_tokens_est", "mean"), max_tokens_est=("prompt_tokens_est", "max")
).round(0)

In [ ]:
total_tokens = d["prompt_tokens_est"].sum()
prefix_tokens = len(prefixes.iloc[0]) / 3.5
print(f"total input tokens (est.): {total_tokens / 1e6:.2f} M")
print(f"shared template prefix: ~{prefix_tokens:.0f} tokens per prompt, "
      f"{prefix_tokens * len(d) / total_tokens:.0%} of all input tokens (cacheable with prompt caching)")

### Plot: prompt size

Left: the distribution of estimated prompt tokens across all prompts. Right: prompt size against history length. Every point is one prompt; size grows linearly, because each extra observation adds a fixed-width record.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

ax1.hist(d["prompt_tokens_est"], bins=40, color=C_NO_RI, edgecolor=SURFACE, linewidth=1)
ax1.set(title="Estimated tokens per prompt", xlabel="tokens (est.)", ylabel="prompts")
ax1.grid(axis="x", visible=False)

ax2.scatter(d["n_history_rows"], d["prompt_tokens_est"], s=14, color=C_NO_RI, alpha=0.5, linewidths=0)
ax2.set(title="Prompt size vs history length", xlabel="history rows in prompt", ylabel="tokens (est.)")

fig.tight_layout()
plt.show()

## Per-storm summary

One row per storm: number of prompts, number of RI rows, and when the first RI row occurs.

In [ ]:
storms = d.groupby("sid").agg(
    storm_group=("storm_group", "first"),
    prompts=("sid", "size"),
    ri_rows=(TARGET, "sum"),
    first_ri_obs=("obs_index", lambda s: s[d.loc[s.index, TARGET] == 1].min()),
    max_tokens_est=("prompt_tokens_est", "max"),
).sort_values(["storm_group", "ri_rows"], ascending=[False, False])

print(storms.groupby("storm_group")["prompts"].describe().round(1))
storms.head(15)

### Plot: storm length and RI rate by position

Left: how many prompts (observations) each storm contributes, split by storm group. RI storms tend to be longer. Right: the share of RI rows at each history length, which shows where in a storm's life RI labels fall. Bars are labelled with the number of prompts in each bin.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

bins_storm = range(0, storms["prompts"].max() + 5, 4)
for group, colour, label in [("no_ri", C_NO_RI, "no-RI storms"), ("ri", C_RI, "RI storms")]:
    ax1.hist(storms.loc[storms["storm_group"] == group, "prompts"], bins=bins_storm,
             histtype="step", linewidth=2, color=colour, label=label)
ax1.set(title="Prompts per storm", xlabel="prompts (observations) in storm", ylabel="storms")
ax1.legend()

pos_bins = [0, 4, 8, 12, 16, 20, 30, max(60, int(d["n_history_rows"].max()))]   # last bin reaches the longest history
by_pos = d.groupby(pd.cut(d["n_history_rows"], pos_bins), observed=True)[TARGET].agg(["mean", "size"])
x = range(len(by_pos))
bars = ax2.bar(x, by_pos["mean"], width=0.6, color=C_RI)
ax2.bar_label(bars, labels=[f"n={n}" for n in by_pos["size"]], padding=3, color=INK_MUTED, fontsize=8)
ax2.set_xticks(list(x), [f"{iv.left + 1}-{iv.right}" for iv in by_pos.index])
ax2.set(title="Share of RI rows by history length", xlabel="history rows in prompt", ylabel="RI share")
ax2.yaxis.set_major_locator(plt.MultipleLocator(0.05))
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax2.grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

## Label timeline for one storm

Shows the RI label at each observation of a storm, together with `usa_wind` from the last history row of each prompt. `#` marks an RI row and `.` a non-RI row.

In [ ]:
def storm_timeline(sid):
    g = d[d["sid"] == sid]
    current = [history_records(p)[-1] for p in g["prompt"]]
    out = pd.DataFrame({
        "obs_index": g["obs_index"].values,
        "iso_time": g["iso_time"].values,
        "hours_since_start": [r["hours_since_start"] for r in current],
        "usa_wind": [r["usa_wind"] for r in current],
        TARGET: g[TARGET].values,
    })
    print(sid, "labels:", "".join("#" if y else "." for y in out[TARGET]))
    return out

sid = storms[storms["storm_group"] == "ri"].index[0]   # change to any sid
storm_timeline(sid)

### Plot: storm timeline

`usa_wind` over the storm's life, read from the current row of each prompt. Orange markers are observations labelled RI: from these points, the wind rises by 30 kt or more over the next 24 hours.

In [ ]:
def plot_storm_timeline(sid):
    t = storm_timeline(sid)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t["hours_since_start"], t["usa_wind"], color=C_NO_RI, label="usa_wind", zorder=2)
    ri = t[t[TARGET] == 1]
    ax.scatter(ri["hours_since_start"], ri["usa_wind"], s=45, color=C_RI, edgecolors=SURFACE, linewidths=2,
               label=f"{TARGET} = 1", zorder=3)
    ax.set(title=f"Storm {sid}", xlabel="hours since start", ylabel="usa_wind (kt)")
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()

plot_storm_timeline(sid)

## Read a prompt

`show_prompt` prints one prompt. Pass `history_only=True` to skip the fixed instructions and show only the storm history.

In [ ]:
def show_prompt(sid, obs_index, history_only=False):
    rec = d[(d["sid"] == sid) & (d["obs_index"] == obs_index)].iloc[0]
    p = rec["prompt"]
    print(f"sid={sid} obs_index={obs_index} {TARGET}={rec[TARGET]} tokens_est={rec['prompt_tokens_est']}\n")
    print(p[p.index("<storm_history>"):] if history_only else p)

show_prompt(sid, 0)

In [ ]:
# The prompt at this storm's first RI-labelled observation (RI happens in the next 24 h), history only
first_ri = int(storms.loc[sid, "first_ri_obs"])
show_prompt(sid, first_ri, history_only=True)

## Current-observation features

The features of the current (last) row of each prompt, as a dataframe. Compare RI and non-RI rows, and see how often each feature is missing.

In [ ]:
current = pd.DataFrame([history_records(p)[-1] for p in d["prompt"]], index=d.index)
current[TARGET] = d[TARGET]

print("missing share per feature (current row):")
print(current.isna().mean().round(3).sort_values(ascending=False).head(10))

In [ ]:
# Median of each feature for RI vs non-RI rows
current.groupby(TARGET).median().T.round(2)

### Plot: key features, RI vs non-RI rows

Distributions of selected current-row features for non-RI (blue) and RI (orange) rows. Boxes span the 25th to 75th percentile, the line inside is the median, and outliers are hidden. Missing values are dropped per feature.

In [ ]:
KEY_FEATURES = [
    ("usa_wind", "kt"), ("usa_wind_delta_tminus24h", "kt"), ("wind_shear_annulus_ms", "m/s"),
    ("sea_surface_temp_200km_mean", "K"), ("column_water_vapour_200km_mean", "kg/m²"), ("cold_cloud_fraction_100km", "fraction"),
    ("dist2land", "km"), ("sea_level_pressure_200km_mean", "Pa"),
]

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, (feat, unit) in zip(axes.flat, KEY_FEATURES):
    groups = [current.loc[current[TARGET] == y, feat].dropna() for y in (0, 1)]
    bp = ax.boxplot(groups, tick_labels=["no RI", "RI"], widths=0.5, showfliers=False, patch_artist=True,
                    medianprops={"color": INK, "linewidth": 2}, whiskerprops={"color": INK_MUTED},
                    capprops={"color": INK_MUTED}, boxprops={"linewidth": 0})
    for box, colour in zip(bp["boxes"], (C_NO_RI, C_RI)):
        box.set_facecolor(colour)
    ax.set_title(feat, fontsize=9, fontweight="bold")
    ax.set_ylabel(unit)
    ax.grid(axis="x", visible=False)

fig.suptitle("Current-row features: non-RI vs RI rows", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
plt.show()